# 02 · Shared data preprocessing

Prepare the **humanitarian `all_data_en`** splits for TF-IDF + Logistic Regression and BERT. Keep all 11 labels and original split assignments. Original TSV files remain unchanged. Both models should use `text_clean` from these outputs, with `class_label` as the target; event metadata is not a model input.

This notebook:
* Loads the humanitarian all_data_en splits
* Cleans URLs, mentions, hashtags, and whitespace
* Removes the one duplicate ID from dev
* Checks labels and duplicate overlap
* Saves the prepared files and reports in data/processed/

In [1]:
from pathlib import Path
import html
import json
import re
import unicodedata
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/raw/all_data_en").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data/raw/all_data_en").is_dir(), "Run from the project root or notebooks/"
RAW = ROOT / "data/raw/all_data_en"
OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
SPLITS = ["train", "dev", "test"]
LABELS = sorted([
    "affected_individual", "caution_and_advice", "displaced_and_evacuations",
    "donation_and_volunteering", "infrastructure_and_utilities_damage",
    "injured_or_dead_people", "missing_and_found_people", "not_humanitarian",
    "requests_or_needs", "response_efforts", "sympathy_and_support",
])
frames = []
for split in SPLITS:
    frame = pd.read_csv(
        RAW / f"crisis_consolidated_humanitarian_filtered_lang_en_{split}.tsv",
        sep="\t", dtype={"id": "string"},
    )
    frame["split"] = split
    frame["source_row"] = range(1, len(frame) + 1)
    frames.append(frame)
data = pd.concat(frames, ignore_index=True)
original_counts = data.groupby("split").size().reindex(SPLITS)
print("Original rows:")
print(original_counts.to_string())
print("\nMissing values (lang_conf is metadata, not required):")
print(data.isna().sum().to_string())


Original rows:
split
train    61164
dev       8935
test     17356

Missing values (lang_conf is metadata, not required):
id                 0
event              0
source             0
text               0
lang               0
lang_conf      37401
class_label        0
split              0
source_row         0


## 1. Inspect the duplicate ID

IDs are read as strings to preserve their exact values. Inspect repeated IDs before removing anything. Rows with missing language confidence remain usable.


In [2]:
duplicate_ids = data[data["id"].notna() & data.duplicated("id", keep=False)]
print(duplicate_ids[["id", "split", "source", "event", "class_label", "text"]].to_string(index=False))


                id split      source           event      class_label                                                                                                                           text
869989002669166592 train   crisismmd srilanka_floods not_humanitarian MR. BLACK X Ale Mora - Party People is OUT NOW!!! @RevealedRec @mr_black_music @djalemora á½’5á½’5á½’5 https://t.co/xnQeYaTZuc
869989002669166592   dev aidr_system disaster_events not_humanitarian       MR. BLACK X Ale Mora - Party People is OUT NOW!!! @RevealedRec @mr_black_music @djalemora ὒ5ὒ5ὒ5 https://t.co/xnQeYaTZuc


## 2. Light text cleaning

Decode HTML entities, normalize Unicode to NFC, replace URLs with `URL` and mentions with `USER`, remove the `#` symbol while keeping hashtag words, and collapse whitespace. Preserve case, negations, numbers, emojis, stopwords, and punctuation. Do not stem or lemmatize. TF-IDF and BERT apply their own tokenization afterward.

Keep `text` as the original and write cleaned text to `text_clean`. Flag possible encoding artifacts instead of guessing repairs that could corrupt valid text. These flags are heuristic and are not a reason to delete a row.

`event_group` removes only a trailing `-ontopic`. It is a starting point for event grouping: other aliases (such as West Texas names) still need manual review before cross-crisis testing. Generic events like `disaster_events` are not individual held-out disasters.


In [3]:
def clean_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", html.unescape(str(value)))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\w)@[A-Za-z0-9_]+", " USER ", text)
    text = re.sub(r"#(?=\w)", "", text)
    return re.sub(r"\s+", " ", text).strip()

data["text_clean"] = data["text"].map(clean_text)
data["event_group"] = data["event"].str.replace(r"-ontopic$", "", regex=True)
data["possible_encoding_issue"] = data["text"].str.contains(
    r"â€|Ã.|Â.|�|á½", regex=True, na=False
)
print("Possible encoding issues by split (flagged, retained):")
print(data.groupby("split")["possible_encoding_issue"].sum().reindex(SPLITS).to_string())
print("\nTraining examples before / after cleaning:")
print(data.loc[data["split"].eq("train"), ["text", "text_clean"]].head(3).to_string(index=False))


Possible encoding issues by split (flagged, retained):
split
train    4054
dev       588
test     1099

Training examples before / after cleaning:
                                                                                                                                          text                                                                                                                                text_clean
                                                         Approximately 100km long firebreaks have been constructed by Indonesia and elsewhere.                                                     Approximately 100km long firebreaks have been constructed by Indonesia and elsewhere.
                                                                                                      God bless you... https://t.co/AnEy1ydkkz                                                                                                                      God bless you... URL
RT @perrea

## 3. Remove unusable and repeated examples

Apply a fixed policy, without tuning it to model scores:

1. Remove missing/blank text, missing IDs, and invalid labels.
2. If the same ID or identical cleaned text has conflicting labels, exclude every affected row rather than invent a correct label. Record these exclusions for review.
3. For remaining repeated IDs or cleaned texts, keep the first occurrence in **train → dev → test** order. This preserves training examples and removes later copies from evaluation; it changes evaluation sizes and must be reported. No examples move between splits.

Cleaning can turn different posts into the same text, especially short mention/link-only posts. The removal audit makes this effect visible. This only handles exact cleaned-text duplicates, not all near-duplicates or retweets.


In [4]:
data["removal_reason"] = ""
invalid = (
    data["text_clean"].eq("") | data["id"].isna()
    | data["id"].fillna("").str.strip().eq("")
    | ~data["class_label"].isin(LABELS)
)
data.loc[invalid, "removal_reason"] = "missing_or_invalid_required_value"
valid = data.loc[~invalid]
conflict_mask = pd.Series(False, index=valid.index)
for key in ["id", "text_clean"]:
    conflicting = valid.groupby(key)["class_label"].transform("nunique").gt(1)
    conflict_mask |= conflicting

data.loc[conflict_mask.index[conflict_mask], "removal_reason"] = "conflicting_labels_for_id_or_cleaned_text"
remaining = data.loc[data["removal_reason"].eq("")]
repeated_ids = remaining.duplicated("id", keep="first")
data.loc[remaining.index[repeated_ids], "removal_reason"] = "duplicate_id"
remaining = data.loc[data["removal_reason"].eq("")]
repeated_text = remaining.duplicated("text_clean", keep="first")
data.loc[remaining.index[repeated_text], "removal_reason"] = "duplicate_cleaned_text"

removed = data.loc[data["removal_reason"].ne("")].copy()
clean = data.loc[data["removal_reason"].eq("")].drop(columns="removal_reason").copy()
print("Removed rows by reason and split:")
print(pd.crosstab(removed["removal_reason"], removed["split"]).reindex(columns=SPLITS, fill_value=0).to_string())
print("\nRows before and after:")
summary = pd.DataFrame({"original": original_counts,
                        "retained": clean.groupby("split").size().reindex(SPLITS, fill_value=0)})
summary["removed"] = summary["original"] - summary["retained"]
print(summary.to_string())


Removed rows by reason and split:
split           train  dev  test
removal_reason                  
duplicate_id        0    1     0

Rows before and after:
       original  retained  removed
split                             
train     61164     61164        0
dev        8935      8934        1
test      17356     17356        0


## 4. Validate and save

Check required fields, labels, and exact overlap after cleaning. Save counts by class so the effect on smaller categories is visible. These are standard splits; overlapping disaster events are expected here. A separate event-disjoint experiment is still required for cross-crisis generalization.


In [5]:
assert clean["text_clean"].str.strip().ne("").all()
assert clean["class_label"].isin(LABELS).all()
assert not clean["id"].duplicated().any()
assert not clean["text_clean"].duplicated().any()
for split in SPLITS:
    subset = clean.loc[clean["split"].eq(split)]
    assert len(subset) > 0
    assert set(subset["class_label"]) == set(LABELS), f"A class disappeared from {split}"
for left, right in [("train", "dev"), ("train", "test"), ("dev", "test")]:
    for key in ["id", "text_clean"]:
        overlap = set(clean.loc[clean["split"].eq(left), key]) & set(clean.loc[clean["split"].eq(right), key])
        assert not overlap, (left, right, key)
print("Passed: valid labels/text, all 11 classes per split, no repeated IDs or cleaned texts.")

before = pd.crosstab(data["class_label"], data["split"]).reindex(index=LABELS, columns=SPLITS, fill_value=0)
after = pd.crosstab(clean["class_label"], clean["split"]).reindex(index=LABELS, columns=SPLITS, fill_value=0)
class_counts = pd.concat({"before": before, "after": after, "removed": before - after}, axis=1)
print("\nClass counts:")
print(class_counts.to_string())
for split in SPLITS:
    path = OUT / f"humanitarian_{split}.tsv"
    clean.loc[clean["split"].eq(split)].to_csv(path, sep="\t", index=False)
    print(f"Saved {path.relative_to(ROOT)}")
removed.to_csv(OUT / "preprocessing_removed_rows.tsv", sep="\t", index=False)
class_counts.to_csv(OUT / "preprocessing_class_counts.csv")
summary.to_csv(OUT / "preprocessing_split_counts.csv")
report = {
    "source": "data/raw/all_data_en humanitarian train/dev/test",
    "model_input": "text_clean", "target": "class_label", "labels": LABELS,
    "split_priority_for_duplicates": SPLITS,
    "conflict_policy": "exclude all rows sharing an ID or cleaned text with conflicting labels",
    "counts": json.loads(summary.to_json(orient="index")),
    "removal_reasons": {str(k): int(v) for k, v in removed["removal_reason"].value_counts().items()},
    "encoding_policy": "flag suspected artifacts; no automatic repair",
    "cross_crisis_ready": False,
}
(OUT / "preprocessing_report.json").write_text(json.dumps(report, indent=2) + "\n")
# Verify saved TSVs round-trip correctly, preserving IDs and text.
for split in SPLITS:
    saved = pd.read_csv(OUT / f"humanitarian_{split}.tsv", sep="\t", dtype={"id": "string"})
    expected = clean.loc[clean["split"].eq(split)]
    assert saved["id"].tolist() == expected["id"].tolist()
    assert saved["text_clean"].tolist() == expected["text_clean"].tolist()
print("Saved TSV round-trip checks passed.")


Passed: valid labels/text, all 11 classes per split, no repeated IDs or cleaned texts.

Class counts:
                                    before               after              removed         
split                                train   dev   test  train   dev   test   train dev test
class_label                                                                                 
affected_individual                   2452   366    692   2452   366    692       0   0    0
caution_and_advice                    2100   307    583   2100   307    583       0   0    0
displaced_and_evacuations              358    53     99    358    53     99       0   0    0
donation_and_volunteering             5175   761   1452   5175   761   1452       0   0    0
infrastructure_and_utilities_damage   3539   509   1002   3539   509   1002       0   0    0
injured_or_dead_people                1942   270    561   1942   270    561       0   0    0
missing_and_found_people               373    55    103    37

## Next: baseline

Load `humanitarian_train.tsv` and `humanitarian_dev.tsv` from `data/processed/`. Train a scikit-learn pipeline of `TfidfVectorizer` and `LogisticRegression` using **`text_clean` → `class_label`**. Fit TF-IDF on training text only. Compare settings using dev macro-F1 and per-class metrics; reserve test scoring for the final comparison.

BERT should load the same prepared files and tokenize `text_clean`. Keep raw data and audit files for reproducibility. No model is trained in this notebook.
